# 03 · Evaluate — Test set (fold 10)

Loads pre-saved prediction arrays from `02_train.ipynb`. No model loading, no retraining.

**Required artefacts (produced by `02_train.ipynb`):**
- `checkpoints/test_preds.npy` — shape `(N_test, 5)` float32 sigmoid outputs
- `checkpoints/test_targets.npy` — shape `(N_test, 5)` int8 binary targets

Classes in column order: `NORM, MI, STTC, CD, HYP`

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from IPython.display import display, Markdown

/Users/roshani/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/roshani/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
CKPT_DIR = Path("checkpoints")

preds = np.load(CKPT_DIR / "test_preds.npy")
targets = np.load(CKPT_DIR / "test_targets.npy")

assert preds.shape == targets.shape, "Shape mismatch"
assert preds.shape[1] == 5, f"Expected 5 classes, got {preds.shape[1]}"
assert not np.any(np.isnan(preds)), "NaN detected in preds"
assert not np.any(np.isinf(preds)), "Inf detected in preds"

print(f"preds   shape: {preds.shape}  dtype: {preds.dtype}")
print(f"targets shape: {targets.shape}  dtype: {targets.dtype}")
print(f"Positive label counts per class: {dict(zip(CLASSES, targets.sum(axis=0).astype(int)))}")

preds   shape: (2158, 5)  dtype: float32
targets shape: (2158, 5)  dtype: float32
Positive label counts per class: {'NORM': 963, 'MI': 550, 'STTC': 521, 'CD': 496, 'HYP': 262}


## 1 · AUC — per class and macro

In [4]:
per_class_auc = [
    roc_auc_score(targets[:, i], preds[:, i])
    for i in range(len(CLASSES))
]
macro_auc = roc_auc_score(targets, preds, average="macro")
support = targets.sum(axis=0).astype(int)

auc_df = (
    pd.DataFrame({"Class": CLASSES, "AUC": per_class_auc, "Support (positives)": support})
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)
auc_df["AUC"] = auc_df["AUC"].map("{:.4f}".format)

print(f"Macro AUC: {macro_auc:.4f}\n")
print(auc_df.to_string(index=False))


Macro AUC: 0.9081

Class    AUC  Support (positives)
 NORM 0.9388                  963
 STTC 0.9338                  521
   CD 0.9252                  496
   MI 0.9155                  550
  HYP 0.8272                  262


## 2 · Benchmark comparison

In [5]:
# Renders a comparison table using the macro_auc value computed above.
# The published benchmark figure (~0.93) is from Strodthoff et al. 2021,
# Table II, superdiagnostic task, resnet1d_wang column.
comparison_md = f"""
| Model | Macro AUC (superdiagnostic, fold 10) |
|---|---|
| **This repo** (from-scratch ResNet-1D) | **{macro_auc:.4f}** |
| resnet1d_wang — Strodthoff et al. 2021 | ~0.93 |

*This model is a from-scratch reimplementation in the same architecture family as resnet1d_wang
and does not use the original published weights.*

**Citation:** N. Strodthoff, P. Wagner, T. Schaeffter, W. Samek,
"Deep Learning for ECG Analysis: Benchmarks and Insights from PTB-XL,"
IEEE J. Biomed. Health Inform., vol. 25, no. 5, pp. 1519–1528, 2021.
Code: [ecg_ptbxl_benchmarking](https://github.com/helme/ecg_ptbxl_benchmarking)
"""
display(Markdown(comparison_md))


| Model | Macro AUC (superdiagnostic, fold 10) |
|---|---|
| **This repo** (from-scratch ResNet-1D) | **0.9081** |
| resnet1d_wang — Strodthoff et al. 2021 | ~0.93 |

*This model is a from-scratch reimplementation in the same architecture family as resnet1d_wang
and does not use the original published weights.*

**Citation:** N. Strodthoff, P. Wagner, T. Schaeffter, W. Samek,
"Deep Learning for ECG Analysis: Benchmarks and Insights from PTB-XL,"
IEEE J. Biomed. Health Inform., vol. 25, no. 5, pp. 1519–1528, 2021.
Code: [ecg_ptbxl_benchmarking](https://github.com/helme/ecg_ptbxl_benchmarking)


## 3 · Confusion counts at 0.5 threshold

In [7]:
preds_bin = (preds >= 0.5).astype(int)

rows = []
for i, cls in enumerate(CLASSES):
    pos_mask = targets[:, i] == 1
    neg_mask = targets[:, i] == 0
    fp = int(((preds_bin[:, i] == 1) & neg_mask).sum())
    fn = int(((preds_bin[:, i] == 0) & pos_mask).sum())
    rows.append({"Class": cls, "FP": fp, "FN": fn})

conf_df = pd.DataFrame(rows)
print(conf_df.to_string(index=False))


Class  FP  FN
 NORM 156 148
   MI 174 124
 STTC 118 136
   CD  80 159
  HYP  62 162


## 4 · Five test records with the largest total prediction error

In [8]:
total_abs_error = np.abs(preds - targets).sum(axis=1)  # (N_test,)
worst_idx = np.argsort(total_abs_error)[::-1][:5]

worst_rows = []
for rank, idx in enumerate(worst_idx):
    row = {"rank": rank + 1, "record_idx": int(idx)}
    for j, cls in enumerate(CLASSES):
        row[f"true_{cls}"] = int(targets[idx, j])
    for j, cls in enumerate(CLASSES):
        row[f"pred_{cls}"] = round(float(preds[idx, j]), 3)
    row["total_abs_error"] = round(float(total_abs_error[idx]), 3)
    worst_rows.append(row)

worst_df = pd.DataFrame(worst_rows).set_index("rank")
display(worst_df)

,record_idx,true_NORM,true_MI,true_STTC,true_CD,true_HYP,pred_NORM,pred_MI,pred_STTC,pred_CD,pred_HYP,total_abs_error
rank,,,,,,,,,,,,
1,128,0,1,0,1,0,0.021,0.019,0.909,0.014,0.792,3.689
2,1179,0,0,1,1,1,0.716,0.006,0.192,0.089,0.016,3.425
3,611,1,0,1,0,0,0.047,0.850,0.047,0.550,0.094,3.401
4,1418,0,1,0,0,0,0.000,0.377,0.962,0.802,0.763,3.151
5,1537,0,0,0,1,0,0.000,0.916,0.934,0.285,0.548,3.113


## Limitations

1. Model trained from scratch rather than initialised from published pretrained weights, unlike some benchmark results.
2. Single train run with one random seed, no ensembling or cross-validation across seeds.
3. Signals normalised per-record (z-score per lead) rather than using train-set-fitted global statistics, which may shift calibration slightly versus the benchmark protocol.
4. Class imbalance across the five superclasses not corrected in the loss function (no class weighting or resampling), which likely explains lower AUC on minority classes such as HYP.